# 08 トレーディング有用性分析

ベースラインモデル（特徴量追加なし）を対象に、以下を検証する。

1. **モデル評価サマリー** — IC・ICIR・Sharpe・方向的中率
2. **IC 安定性** — フォールド別 IC（GBDT 金融データの標準的検証）
3. **限月別 IC** — M1〜M8 でどの限月が予測しやすいか
4. **MPM 近接別 IC** — 会合直前ほど精度が高いか
5. **限月別 累積 P&L** — 5d モデル、サインフォロー戦略
6. **最新日予測** — 現時点での 3d・5d 予測差分（M1〜M8）

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import spearmanr

from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, calculate_metrics, get_features_and_target

EXCEL_PATH = '../data/BOJ_data.xlsx'
MEETING_CSV_PATH = '../data/BOJ_meeting_history.csv'
START_DATE = '2024-01-01'

# ベースライン（特徴量追加なし）
df_raw = load_and_clean_data(EXCEL_PATH, MEETING_CSV_PATH)
df_feat = generate_features(df_raw)
df_pooled = pool_boj_data(df_feat)

print(f'Pooled data: {df_pooled.shape}')
print(f'Date range : {df_pooled["Date"].min().date()} to {df_pooled["Date"].max().date()}')
print(f'OOS start  : {START_DATE}')

In [ ]:
print('Running 3d walk-forward validation...')
res_3d, model_3d, X_test_3d, y_test_3d, X_train_3d = walk_forward_validation(
    df_pooled, 'Target_3d', START_DATE, return_model=True)

print('Running 5d walk-forward validation...')
res_5d, model_5d, X_test_5d, y_test_5d, X_train_5d = walk_forward_validation(
    df_pooled, 'Target_5d', START_DATE, return_model=True)

# Days_to_MPM をマージ（後の分析用）
dtm_map = df_pooled[['Date', 'Meeting_Index', 'Days_to_MPM']].drop_duplicates()
res_3d = res_3d.merge(dtm_map, on=['Date', 'Meeting_Index'], how='left')
res_5d = res_5d.merge(dtm_map, on=['Date', 'Meeting_Index'], how='left')

print(f'\nOOS samples — 3d: {len(res_3d)}, 5d: {len(res_5d)}')
print(f'Folds       — 3d: {res_3d["Fold"].nunique()}, 5d: {res_5d["Fold"].nunique()}')

## 1. モデル評価サマリー

金融 GBDT の標準的指標：IC（Spearman）、ICIR（IC の安定性）、方向的中率、戦略 Sharpe、最大ドローダウン

In [ ]:
def compute_summary(res, label):
    m = calculate_metrics(res['Actual'], res['Pred'])
    fold_ics = res.groupby('Fold').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0])
    icir = fold_ics.mean() / fold_ics.std() if fold_ics.std() > 0 else np.nan
    pnl = np.sign(res['Pred']) * res['Actual']
    sharpe = pnl.mean() / pnl.std() * np.sqrt(252) if pnl.std() > 0 else np.nan
    cum = pnl.cumsum()
    max_dd = (cum.cummax() - cum).max()
    return {
        'Target'               : label,
        'IC (Spearman)'        : round(m['IC'], 4),
        'ICIR'                 : round(icir, 4),
        'Direction Acc.'       : f"{m['Direction_Accuracy']:.1%}",
        'Dir Acc. (Large Move)': f"{m['Direction_Accuracy_LargeMove']:.1%}",
        'Strategy Sharpe'      : round(sharpe, 3),
        'Max Drawdown'         : round(max_dd, 6),
        'Folds'                : int(res['Fold'].nunique()),
        'OOS Samples'          : len(res),
    }

df_summary = pd.DataFrame([
    compute_summary(res_3d, '3d'),
    compute_summary(res_5d, '5d'),
])
print('=== モデル評価サマリー ===')
print(df_summary.to_string(index=False))

## 2. IC 安定性（フォールド別）

GBDT 金融データ分析の核心的検証。IC が時系列で安定しているか（レジーム依存でないか）を確認する。
全フォールド正 かつ ばらつきが小さい ほど実運用に適したモデル。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('IC by Fold — Walk-Forward OOS Stability', fontsize=13)

for ax, res, label in [(axes[0], res_3d, 'Target_3d'), (axes[1], res_5d, 'Target_5d')]:
    fold_ics = res.groupby('Fold').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0])
    fold_start = res.groupby('Fold')['Date'].min().dt.strftime('%Y-%m')
    colors = ['steelblue' if v > 0 else 'salmon' for v in fold_ics]
    ax.bar(range(len(fold_ics)), fold_ics.values, color=colors,
           edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(fold_ics.mean(), color='navy', linestyle='--', linewidth=1.2,
               label=f'Mean={fold_ics.mean():.3f}  ICIR={fold_ics.mean()/fold_ics.std():.2f}')
    ax.set_xticks(range(len(fold_ics)))
    ax.set_xticklabels(fold_start.values, rotation=45, ha='right', fontsize=8)
    ax.set_title(label)
    ax.set_ylabel('IC (Spearman)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. 限月別 IC（M1〜M8）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('IC by Tenor (Meeting Index)', fontsize=13)

for ax, res, label in [(axes[0], res_3d, 'Target_3d'), (axes[1], res_5d, 'Target_5d')]:
    mi_ics = res.groupby('Meeting_Index').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0])
    xlabels = [f'M{i}' for i in mi_ics.index]
    colors = ['steelblue' if v > 0 else 'salmon' for v in mi_ics]
    ax.bar(xlabels, mi_ics.values, color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(mi_ics.mean(), color='navy', linestyle='--', linewidth=1.2,
               label=f'Mean={mi_ics.mean():.3f}')
    for j, (v, lbl) in enumerate(zip(mi_ics.values, xlabels)):
        ax.text(j, v + 0.005 * np.sign(v if v != 0 else 1),
                f'{v:.3f}', ha='center', fontsize=8)
    ax.set_title(label)
    ax.set_ylabel('IC (Spearman)')
    ax.set_xlabel('Tenor')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. MPM 近接別 IC（Days_to_MPM バケット）

会合直前（≤5日）は大きなレート変動が予測しやすいか確認する。

In [ ]:
def dtm_bucket(d):
    if pd.isna(d): return 'Unknown'
    if d <= 5:  return '≤5  (直前)'
    if d <= 15: return '6-15'
    return '16+'

BUCKET_ORDER = ['≤5  (直前)', '6-15', '16+']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('IC by Days to MPM', fontsize=13)

for ax, res, label in [(axes[0], res_3d, 'Target_3d'), (axes[1], res_5d, 'Target_5d')]:
    r = res.copy()
    r['bucket'] = r['Days_to_MPM'].apply(dtm_bucket)
    bkt_ics = r.groupby('bucket').apply(
        lambda g: spearmanr(g['Actual'], g['Pred'])[0]).reindex(BUCKET_ORDER)
    bkt_n = r.groupby('bucket').size().reindex(BUCKET_ORDER)
    colors = ['steelblue' if v > 0 else 'salmon' for v in bkt_ics]
    ax.bar(BUCKET_ORDER, bkt_ics.values, color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    for j, (v, bkt) in enumerate(zip(bkt_ics.values, BUCKET_ORDER)):
        n = bkt_n[bkt]
        ax.text(j, v + 0.005 * np.sign(v if v != 0 else 1),
                f'{v:.3f}\n(n={n})', ha='center', fontsize=9)
    ax.set_title(label)
    ax.set_ylabel('IC (Spearman)')
    ax.set_xlabel('Days to Next MPM')

plt.tight_layout()
plt.show()

## 5. 限月別 累積 P&L（5d モデル）

**戦略ルール**：予測が正（上昇）ならロング、負（下落）ならショート。
累積 P&L = Σ sign(Pred) × Actual（rate change）

単位はそのまま（OIS スプレッド変化、例：0.01 = 1bps）。

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
fig.suptitle('Cumulative P&L by Tenor — 5d Model (OOS, Sign-Follow Strategy)', fontsize=13)

for i, mi in enumerate(range(1, 9)):
    t = res_5d[res_5d['Meeting_Index'] == mi].sort_values('Date').copy()
    t['PnL']    = np.sign(t['Pred']) * t['Actual']
    t['CumPnL'] = t['PnL'].cumsum()

    total   = t['CumPnL'].iloc[-1]
    sharpe  = (t['PnL'].mean() / t['PnL'].std() * np.sqrt(252)
               if t['PnL'].std() > 0 else np.nan)
    roll_max = t['CumPnL'].cummax()
    max_dd   = (roll_max - t['CumPnL']).max()
    hit_rate = (t['PnL'] > 0).mean()

    ax = axes[i]
    ax.plot(t['Date'], t['CumPnL'], color='steelblue', linewidth=1.2)
    ax.fill_between(t['Date'], t['CumPnL'], 0,
                    where=t['CumPnL'] >= 0, alpha=0.15, color='steelblue')
    ax.fill_between(t['Date'], t['CumPnL'], 0,
                    where=t['CumPnL'] <  0, alpha=0.15, color='salmon')
    ax.axhline(0, color='black', linewidth=0.6)

    stats_text = (f'Total={total:.4f}\nSharpe={sharpe:.2f}\n'
                  f'MaxDD={max_dd:.4f}\nHit={hit_rate:.1%}')
    ax.set_title(f'M{mi}', fontsize=10, fontweight='bold')
    ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
            va='top', fontsize=7.5, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Cum P&L', fontsize=8)

plt.tight_layout()
plt.show()

## 6. 最新日予測：3d・5d 予測差分（M1〜M8）

最終フォールドのモデルを使い、最新データ日時点での将来予測を可視化する。
横軸：限月（M1〜M8）、縦軸：予測される OIS スプレッド変化。

In [ ]:
# 最終フォールドのモデルが使う特徴量名
feat_cols_3d = model_3d.feature_name()
feat_cols_5d = model_5d.feature_name()

# 最新日の行を取得（特徴量は揃っている前提）
latest_date = df_pooled['Date'].max()
latest_rows = df_pooled[df_pooled['Date'] == latest_date].sort_values('Meeting_Index').copy()
tenors = [f'M{int(mi)}' for mi in latest_rows['Meeting_Index'].values]

pred_3d = model_3d.predict(latest_rows[feat_cols_3d])
pred_5d = model_5d.predict(latest_rows[feat_cols_5d])

# bps 変換
pred_3d_bps = pred_3d * 10000
pred_5d_bps = pred_5d * 10000

x = np.arange(len(tenors))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars3 = ax.bar(x - width/2, pred_3d_bps, width, label='3d prediction',
               color='steelblue', edgecolor='black', linewidth=0.5)
bars5 = ax.bar(x + width/2, pred_5d_bps, width, label='5d prediction',
               color='coral',     edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)

for bar in list(bars3) + list(bars5):
    h = bar.get_height()
    va = 'bottom' if h >= 0 else 'top'
    offset = 0.3 if h >= 0 else -0.3
    ax.text(bar.get_x() + bar.get_width()/2, h + offset,
            f'{h:.1f}', ha='center', va=va, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(tenors)
ax.set_xlabel('Tenor')
ax.set_ylabel('Predicted Change (bps)')
ax.set_title(
    f'Latest Date Predictions as of {latest_date.date()}\n'
    f'3-day & 5-day predicted OIS spread change by tenor'
)
ax.legend()
plt.tight_layout()
plt.show()

# テーブルでも表示
df_pred = pd.DataFrame({
    'Tenor': tenors,
    'Days_to_MPM': latest_rows['Days_to_MPM'].values,
    'Pred_3d (bps)': pred_3d_bps.round(2),
    'Pred_5d (bps)': pred_5d_bps.round(2),
})
print(f'\n最新日 ({latest_date.date()}) 予測値')
print(df_pred.to_string(index=False))